In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import torch
import matplotlib.pyplot as plt

# Prepare ice flux (q) observation tensor

Create a tensor with columns [x_norm, y_norm, flux_x_scaled, flux_y_scaled]

In [ ]:
# import region bounds
from regions import ROSS_ISLAND_BOUNDS

# assign region bounds
x_min = ROSS_ISLAND_BOUNDS["x_min"]
x_max = ROSS_ISLAND_BOUNDS["x_max"]
y_min = ROSS_ISLAND_BOUNDS["y_min"]
y_max = ROSS_ISLAND_BOUNDS["y_max"]

## Load cleaned thickness point data

In [ ]:
# Load thickness points
# NOTE: Outliers are removed from this file.
thickness = pd.read_csv("data/bedmap123_ross_island_cleaned_237.csv")
thickness.describe()

## Load raw ice velocity data

- 450 m nominal resolution
- interpolate at point locations

In [ ]:
# Load ice velocity data (raw)
# TODO define local path to ice velocity data
ice_vel = xr.load_dataset("/home/kim/data/nsidc/antarctic_ice_vel_phase_map_v01.nc")
# ice_vel = xr.load_dataset("nsidc/antarctic_ice_vel_phase_map_v01.nc")

# crop
velocity = ice_vel.sel(
    x = slice(x_min, x_max), 
    y = slice(y_max, y_min))

In [ ]:
# Interpolate
vel_at_points = velocity[["VX", "VY"]].interp(
    x = ("points", thickness["x"].values),
    y = ("points", thickness["y"].values),
    method = "linear",
)

# Add to dataset
thickness["VX"] = vel_at_points["VX"].values
thickness["VY"] = vel_at_points["VY"].values

# 240k
print(thickness.shape)

In [ ]:
# Multiply velocity components by thickness (t_fa) to get flux components
thickness["VX_flux"] = thickness["VX"] * thickness["t_fa"]
thickness["VY_flux"] = thickness["VY"] * thickness["t_fa"]

# If you also want total flux magnitude
thickness["flux_mag"] = np.hypot(thickness["VX_flux"], thickness["VY_flux"])

# Optionally drop rows with NaNs if you want a clean dataset
# Check for NaNs
print(thickness[["VX_flux", "VY_flux", "flux_mag"]].isna().sum())

# Remove so that 240k points remain
thickness = thickness.dropna(subset = ["VX_flux", "VY_flux", "flux_mag"])
print(thickness.shape)

## Load MODIS MOA for visualisation backrgound

In [ ]:
modis_ross = torch.load("data/modis/moa125_2014_hp1_v01_ross_with_grid_no_ocean.pt", weights_only = False)

In [ ]:
# Use same subsample rate as the ice thickness visualisations
SUBSAMPLE_RATE = 10

fig, ax = plt.subplots(figsize = (8, 6))

# 1) MODIS as background (draw first)
ax.pcolormesh(
    modis_ross[0],
    modis_ross[1], 
    modis_ross[2],
    cmap = "gray",
    # softer greys
    vmin = -30_000, 
    vmax = 30_000,
    # as less saturated background
    alpha = 0.4,
    zorder = 0,
)

# 2) SMB on top (draw second)
ax.scatter(
    thickness["x"][::SUBSAMPLE_RATE], 
    thickness["y"][::SUBSAMPLE_RATE], 
    c = thickness["flux_mag"][::SUBSAMPLE_RATE], 
    s = 0.6, 
    alpha = 0.8, 
    cmap = "turbo",
    vmin = 0,
    # 400 k worked well
    vmax = 400_000 
    )

ax.set_aspect("equal")
ax.set_axis_off()

# crop
ax.set_ylim(y_min, y_max)
ax.set_xlim(x_min, x_max)

fig.savefig("figures/ice_flux_ross_island.png", dpi = 300, bbox_inches = "tight", pad_inches = 0)
plt.show()

## cmap

In [ ]:
cmap = "turbo"
dpi = 300

# 1 x N gradient
grad = np.linspace(0, 1, 2048)[None, :]

fig = plt.figure(figsize = (6, 0.6), dpi = dpi)
ax = fig.add_axes([0, 0, 1, 1]) 
ax.imshow(grad, aspect = "auto", cmap = cmap)
ax.set_axis_off()

fig.savefig("figures/cmap/turbo_bar.png", dpi = dpi, transparent = True, bbox_inches = "tight", pad_inches = 0)
plt.close(fig)

# Visualise sparse raw flux vectors - region

(for presentation)

In [ ]:
thickness

# 50 x 50 km zoomed in region
x_min_zoom = 200_000
x_max_zoom = 250_000

y_min_zoom = - 1_360_000
y_max_zoom = - 1_310_000

thickness_zoom = thickness[
    (thickness["x"] >= x_min_zoom) &
    (thickness["x"] <= x_max_zoom) &
    (thickness["y"] >= y_min_zoom) &
    (thickness["y"] <= y_max_zoom)
]

thickness_zoom

In [ ]:
SUBSAMPLE_RATE = 50

fig, ax = plt.subplots(figsize = (8, 6))

# 2) SMB on top (draw second)
ax.scatter(
    thickness_zoom["x"][::SUBSAMPLE_RATE], 
    thickness_zoom["y"][::SUBSAMPLE_RATE], 
    c = thickness_zoom["flux_mag"][::SUBSAMPLE_RATE], 
    s = 1.0, 
    alpha = 0.8, 
    cmap = "turbo",
    vmin = 0,
    # 400 k worked well
    vmax = 400_000 
    )

ax.set_aspect("equal")
ax.set_axis_off()
fig.savefig("figures/ice_flux_ross_island_zoom.png", dpi = 300, bbox_inches = "tight", pad_inches = 0)
plt.show()

In [ ]:
SUBSAMPLE_RATE = 140

fig, ax = plt.subplots(figsize = (8, 6))

ax.pcolormesh(
    modis_ross[0],
    modis_ross[1], 
    modis_ross[2],
    cmap = "gray",
    # softer greys
    vmin = -30_000, 
    vmax = 30_000,
    # as less saturated background
    alpha = 0.4,
    zorder = 0,
)

ax.quiver(
    thickness_zoom["x"][::SUBSAMPLE_RATE], 
    thickness_zoom["y"][::SUBSAMPLE_RATE], 
    thickness_zoom["VX_flux"][::SUBSAMPLE_RATE], 
    thickness_zoom["VY_flux"][::SUBSAMPLE_RATE],
    scale = 7_000_000.0,
    # width = 0.005,
)

ax.set_ylim(y_min_zoom, y_max_zoom)
ax.set_xlim(x_min_zoom, x_max_zoom)

ax.set_aspect("equal")
ax.set_axis_off()
fig.savefig("figures/ice_flux_ross_island_zoom.png", dpi = 300, bbox_inches = "tight", pad_inches = 0)
plt.show()

## Histogram of flux magnitude

Observation: most values are under 400k.
We thus normalise by 100k.

In [ ]:
plt.figure(figsize = (8, 6))
plt.hist(thickness["flux_mag"], bins = 50, edgecolor=  "white")
plt.xlabel("Flux magnitude")
plt.ylabel("Count")
plt.title("Histogram of Ice Flux Magnitude")
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

## Affirm normalisation scalar

In [ ]:
# Print variance after normalisation
print(np.var(thickness["VX_flux"] / 100_000))
print(np.var(thickness["VY_flux"] / 100_000))

## Normalise

Example: 425 meter ice * 438 meter per year (mean)  
425 * 438 = 186_150 m^2/yr

In [ ]:
# Normalize into [0, 1]
thickness["x_norm"] = (thickness["x"] - x_min) / (x_max - x_min)
thickness["y_norm"] = (thickness["y"] - y_min) / (y_max - y_min)

# Build tensor with x_norm, y_norm, scaled flux
flux_tensor = torch.tensor(
    np.stack([
        thickness["x_norm"].values,
        thickness["y_norm"].values,
        thickness["VX_flux"].values / 100_000,
        thickness["VY_flux"].values / 100_000,
    ], axis = 1),
    dtype = torch.float32
)

print(flux_tensor.shape)
# preview first rows
print(flux_tensor[:5])

### Histogram after normalisation

In [ ]:
# Extract the last two columns
vx_flux_scaled = flux_tensor[:, 2].numpy()
vy_flux_scaled = flux_tensor[:, 3].numpy()

# Plot histograms
fig, axes = plt.subplots(1, 2, figsize = (12, 5), sharey = True)

axes[0].hist(vx_flux_scaled, bins = 50, edgecolor = "white")
axes[0].set_title("VX_flux / 100_000")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Count")
axes[0].grid(alpha = 0.3)

axes[1].hist(vy_flux_scaled, bins = 50, edgecolor = "white", color = "orange")
axes[1].set_title("VY_flux / 100000")
axes[1].set_xlabel("Value")
axes[1].grid(alpha = 0.3)

plt.tight_layout()
plt.show()

# Export tensor with all flux observations

In [ ]:
torch.save(flux_tensor, "data/flux_tensor.pt")